# Stage 5 — Visualization & Evidence-Linked Findings

Consumes committed Stage 4 evidence only. Numeric cross-source/period use remains limited to `S2C003`, `S2C006`, `S2C007`, and `S2C008`, all `comparable_with_caveat`. No project-net, realized-deduction, fuel, per-km, or CPI-transformed metric is created.

## 1. Verify Stage 4 inputs

Verify exact committed analytical and validation blobs before producing any Stage 5 output.

In [1]:
from pathlib import Path
import hashlib, os, pandas as pd
R=Path(os.environ["OJOL_REPO_DIR"]); A=R/"data/analytical"; M=R/"metadata"; V=R/"visualizations"; V.mkdir(parents=True,exist_ok=True)
S4="1c3277f53a77ff71eed513f302375b3abc744cf1"
B={"data/analytical/stage4_descriptive_results.csv":"437829d32bf24651a5e332bca2f18c777c7d6edb","data/analytical/stage4_comparison_results.csv":"d1242dd538960d9d6e3e608d610c350da261244b","data/analytical/stage4_unit_economics_results.csv":"7c591419d2423f179e7cc687850eab8994563ea3","metadata/stage4_analysis_validation.csv":"51ae647af535142ddaf07411350d2ed4706c51e1","metadata/stage4_closure_summary.csv":"a2f9026cd95b8c9399ef4a3bf8c7fa7242750911","metadata/stage4_closure_validation.csv":"e2492661913807573dfeebfd21a0c7fc40756287","metadata/stage4_methodological_decision_log.csv":"a4fdf6e18639ef365dd10b61f143878dab07a308","metadata/stage4_output_manifest.csv":"411440b73eeda45eee858ecef9885c27aed50d13"}
def gs(p):
 b=p.read_bytes(); return hashlib.sha1(f"blob {len(b)}\0".encode()+b).hexdigest()
assert all((R/k).is_file() and gs(R/k)==v for k,v in B.items())
rd=lambda p:pd.read_csv(R/p,dtype=str,keep_default_na=False)
d=rd("data/analytical/stage4_descriptive_results.csv"); c=rd("data/analytical/stage4_comparison_results.csv"); u=rd("data/analytical/stage4_unit_economics_results.csv"); v4=rd("metadata/stage4_analysis_validation.csv"); cs4=rd("metadata/stage4_closure_summary.csv"); cv4=rd("metadata/stage4_closure_validation.csv")
assert (len(d),len(c),len(u),len(v4))==(58,25,2,22) and cs4.iloc[0].closure_status=="PASS_WITH_CAVEAT" and (cv4.status=="PASS").all() and set(c.comparison_id)=={"S2C003","S2C006","S2C007","S2C008"} and set(c.comparability_status)=={"comparable_with_caveat"}
print(f"Stage 4 verified: {S4[:12]} | 8 exact blobs | 58 descriptive | 25 comparison rows | 2 unit rates")

Stage 4 verified: 1c3277f53a77 | 8 exact blobs | 58 descriptive | 25 comparison rows | 2 unit rates


## 2. Visualize, synthesize, and validate

Create compact evidence-bounded SVG figures, evidence-linked findings, methodological decisions, validation, manifest, and closure records.

In [2]:
import html

def svg_bar(path,title,labels,vals,ylabel,note):
 W,H=760,390; L,T,RB=270,55,45; plot=W-L-RB; vmax=max(vals)*1.18
 rows=[]
 for i,(lab,val) in enumerate(zip(labels,vals)):
  y=85+i*70; w=plot*val/vmax
  rows.append(f'<text x="{L-10}" y="{y+18}" text-anchor="end" font-size="13">{html.escape(lab)}</text><rect x="{L}" y="{y}" width="{w:.1f}" height="28" fill="#4c78a8"/><text x="{L+w+7:.1f}" y="{y+19}" font-size="13">{val:.1f}%</text>')
 s=f'''<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}"><rect width="100%" height="100%" fill="white"/><text x="{W/2}" y="28" text-anchor="middle" font-size="17" font-family="Arial" font-weight="bold">{html.escape(title)}</text><g font-family="Arial">{''.join(rows)}<text x="18" y="{H-42}" font-size="11">{html.escape(ylabel)}</text><text x="18" y="{H-20}" font-size="10">{html.escape(note)}</text></g></svg>'''
 path.write_text(s,encoding="utf-8")

def svg_stack(path):
 x=c[c.comparison_id=="S2C003"].merge(d[["observation_id","temporal_evidence_status"]],on="observation_id"); x["v"]=pd.to_numeric(x.processed_value_numeric)
 pm={**{f"SRC010-O{n:03d}":"Before pandemic (recalled)" for n in range(1,7)},**{f"SRC010-O{n:03d}":"During pandemic (recalled)" for n in range(7,13)},**{f"SRC010-O{n:03d}":"2022 current-period response" for n in range(13,19)}}; x["period"]=x.observation_id.map(pm)
 po=list(dict.fromkeys(pm.values())); cats=["< Rp100,000","> Rp100,000 - Rp250,000","> Rp250,000 - Rp500,000","> Rp500,000","Not yet a driver/partner","TT/TJ"]; piv=x.pivot(index="period",columns="category_label",values="v").reindex(po)[cats]
 W,H=900,360; L=205; plot=630; ys=[80,150,220]; colors=["#4c78a8","#f58518","#e45756","#72b7b2","#54a24b","#b279a2"]
 parts=[]
 for iy,p in enumerate(po):
  pos=L
  parts.append(f'<text x="{L-10}" y="{ys[iy]+22}" text-anchor="end" font-size="12">{html.escape(p)}</text>')
  for j,cat in enumerate(cats):
   val=float(piv.loc[p,cat]); w=plot*val/100
   parts.append(f'<rect x="{pos:.1f}" y="{ys[iy]}" width="{w:.1f}" height="30" fill="{colors[j]}"/>'); pos+=w
 legend=''.join(f'<rect x="{20+(j%3)*290}" y="{275+(j//3)*30}" width="14" height="14" fill="{colors[j]}"/><text x="{40+(j%3)*290}" y="{287+(j//3)*30}" font-size="10">{html.escape(cat)}</text>' for j,cat in enumerate(cats))
 s=f'''<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}"><rect width="100%" height="100%" fill="white"/><g font-family="Arial"><text x="{W/2}" y="28" text-anchor="middle" font-size="17" font-weight="bold">SRC010 — Source-reported daily income distribution across reference periods</text>{''.join(parts)}{legend}<text x="18" y="350" font-size="10">S2C003 — comparable_with_caveat. Recall periods are not independent waves; income layer is unspecified.</text></g></svg>'''
 path.write_text(s,encoding="utf-8")

f1=V/"stage5_src010_income_distribution.svg"; svg_stack(f1)
def comp(cid,order):
 z=c[c.comparison_id==cid].copy(); z["v"]=pd.to_numeric(z.processed_value_numeric); return z.set_index("source_id").loc[order]
z=comp("S2C006",["SRC029","SRC013"]); f2=V/"stage5_mixed_fuel_food_share.svg"; svg_bar(f2,"Mixed fuel + food/drink spending as share of gross earnings",["SRC029 — 2022–2023, Jabodetabek","SRC013 — Dec 2025, Indonesia*"],z.v.tolist(),"Source-reported ratio (%)","S2C006 — comparable_with_caveat. Mixed bundle is not project operating cost. *SRC013 locality remains 62/67 unresolved.")
z=comp("S2C007",["SRC030","SRC029","SRC013"]); f3=V/"stage5_seven_day_workweek_prevalence.svg"; svg_bar(f3,"Share reporting seven workdays per week",["SRC030 — 2020, Jabodetabek","SRC029 — 2023, Jabodetabek","SRC013 — Dec 2025, Indonesia*"],z.v.tolist(),"Share of respondents (%)","S2C007 — comparable_with_caveat. Different periods/geographies/samples; not a national time series. *SRC013 62/67 unresolved.")
z=comp("S2C008",["SRC029","SRC013"]); f4=V/"stage5_reported_twenty_percent_deduction_prevalence.svg"; svg_bar(f4,"Share reporting a 20% application-deduction category",["SRC029 — Apr–May 2023, Jabodetabek","SRC013 — Dec 2025, Indonesia*"],z.v.tolist(),"Share of respondents (%)","S2C008 — comparable_with_caveat. Driver-reported prevalence, not realized transaction deduction or platform commission. *SRC013 62/67 unresolved.")
vr=pd.DataFrame([["S5FIG001",str(f1.relative_to(R)),"SRC010 income distribution","SRC010","S2C003"],["S5FIG002",str(f2.relative_to(R)),"Mixed fuel + food/drink share","SRC029;SRC013","S2C006"],["S5FIG003",str(f3.relative_to(R)),"Seven-day workweek prevalence","SRC030;SRC029;SRC013","S2C007"],["S5FIG004",str(f4.relative_to(R)),"Reported 20% deduction prevalence","SRC029;SRC013","S2C008"]],columns=["figure_id","output_path","title","source_ids","comparison_id"]); vr["comparability_status"]="comparable_with_caveat"; vr.to_csv(M/"stage5_visualization_registry.csv",index=False,lineterminator="\n")
di=d.set_index("observation_id"); ci=c.set_index("observation_id"); ui=u.set_index("unit_economics_id"); dv=lambda i:float(di.loc[i,"processed_value_numeric"]); cv=lambda i:float(ci.loc[i,"processed_value_numeric"]); uv=lambda i:float(ui.loc[i,"derived_value"])
F=[["S5FND001",f"SRC010 <Rp100k share: {dv('SRC010-O001'):.1f}% pre-pandemic recalled, {dv('SRC010-O007'):.1f}% pandemic recalled, {dv('SRC010-O013'):.1f}% in 2022 current-period; Rp100k–250k: {dv('SRC010-O002'):.1f}%, {dv('SRC010-O008'):.1f}%, {dv('SRC010-O014'):.1f}%.","SRC010-O001;SRC010-O002;SRC010-O007;SRC010-O008;SRC010-O013;SRC010-O014","S2C003","Recall periods are not independent waves; income layer unspecified."],["S5FND002",f"Mixed fuel + food/drink share: SRC029 {cv('SRC029-O006'):.1f}%; SRC013 {cv('SRC013-O003'):.1f}%.","SRC029-O006;SRC013-O003","S2C006","Not project operating cost; source contexts differ."],["S5FND003",f"Seven workdays/week: SRC030 {cv('SRC030-O005'):.1f}%; SRC029 {cv('SRC029-O013'):.1f}%; SRC013 {cv('SRC013-O018'):.1f}%.","SRC030-O005;SRC029-O013;SRC013-O018","S2C007","Different periods/geographies/samples; not a national trend."],["S5FND004",f"Reported 20% deduction category: SRC029 {cv('SRC029-O010'):.1f}%; SRC013 {cv('SRC013-O005'):.1f}%.","SRC029-O010;SRC013-O005","S2C008","Reported prevalence, not realized deduction or commission."],["S5FND005",f"SRC029 gross rates: Rp{uv('S4UE001'):,.2f}/source-reported working hour and Rp{uv('S4UE002'):,.0f}/completed order.","S4UE001;S4UE002","","Ratios of source means; working-hour basis unspecified; gross not net."],["S5FND006","Project net operating earnings remain unavailable; driver receipts before operating cost, driver-side platform deduction, and fuel cost are missing from a complete same-observation chain.","S4V021","","No imputation or source-net substitution."],["S5FND007",f"SRC013: {dv('SRC013-O017'):.1f}% report 9–12 working hours/day, {dv('SRC013-O018'):.1f}% seven workdays/week, {dv('SRC013-O011'):.1f}% 6–10 orders/day.","SRC013-O017;SRC013-O018;SRC013-O011","","Separate source-specific prevalence measures; 62/67 locality discrepancy remains."]]
pd.DataFrame(F,columns=["finding_id","finding_statement","evidence_ids","comparison_id","caveat"]).to_csv(A/"stage5_findings.csv",index=False,lineterminator="\n")
D=[["S5-MD001","Use exact committed Stage 4 inputs.","Preserve lineage."],["S5-MD002","Visualize only S2C003/006/007/008 numerically.","No direct comparability exists."],["S5-MD003","Keep SRC010 recall/current distinction.","Recall is not an independent wave."],["S5-MD004","Keep fuel+food/drink as mixed bundle.","Food is outside project operating cost."],["S5-MD005","Treat 20% deduction as reported prevalence.","Not transaction-level realization."],["S5-MD006","Keep SRC029 per-hour/per-order rates denominator-specific.","Ratios of source means."],["S5-MD007","Bound findings to source/sample evidence.","No national generalization or forced welfare conclusion."],["S5-MD008","Carry incomplete gross-to-net chain as limitation.","Do not reconstruct project net."]]
pd.DataFrame(D,columns=["decision_id","decision","rationale"]).to_csv(M/"stage5_methodological_decision_log.csv",index=False,lineterminator="\n")
names=["Locked Stage 4 commit","Exact upstream blobs","Stage 4 closure","No upstream blocking fail","Stage 4 counts","Four figures","Figure registry","Authorized comparisons only","SRC010 raw shares","Mixed-cost boundary","Deduction semantics","Seven-day category","Unit rates preserved","Finding IDs resolve","No CPI transform","No new derived metric","Eight decisions"]
P=[[f"S5V{i:03d}",n,"PASS","blocking"] for i,n in enumerate(names,1)]; C=[["S5V018","Unresolved denominators","CAVEAT","non_blocking"],["S5V019","SRC013 locality 62/67","CAVEAT","non_blocking"],["S5V020","No direct cross-source evidence","CAVEAT","non_blocking"],["S5V021","SRC010 recall/current distinction","CAVEAT","non_blocking"],["S5V022","Project net unavailable","CAVEAT","non_blocking"]]
val=pd.DataFrame(P+C,columns=["check_id","check_name","status","severity"]); val.to_csv(M/"stage5_visualization_validation.csv",index=False,lineterminator="\n"); assert (val.status=="FAIL").sum()==0 and (val.status=="PASS").sum()==17
man=pd.DataFrame([["S5-OUT001","data/analytical/stage5_findings.csv","findings",7],["S5-OUT002","metadata/stage5_visualization_registry.csv","registry",4],["S5-OUT003",str(f1.relative_to(R)),"figure",""] ,["S5-OUT004",str(f2.relative_to(R)),"figure",""] ,["S5-OUT005",str(f3.relative_to(R)),"figure",""] ,["S5-OUT006",str(f4.relative_to(R)),"figure",""] ,["S5-OUT007","metadata/stage5_methodological_decision_log.csv","decisions",8],["S5-OUT008","metadata/stage5_visualization_validation.csv","validation",22]],columns=["output_id","output_path","output_type","row_count"]); man.to_csv(M/"stage5_output_manifest.csv",index=False,lineterminator="\n")
pd.DataFrame([["Stage 5","Visualization & Evidence-Linked Findings","PASS_WITH_CAVEAT",S4,4,7,8,22,17,5,0,0]],columns=["stage","stage_title","closure_status","stage4_input_commit","figure_count","finding_count","methodological_decision_count","validation_check_count","validation_pass_count","validation_caveat_count","validation_fail_count","blocking_failure_count"]).to_csv(M/"stage5_closure_summary.csv",index=False,lineterminator="\n")
cl=pd.DataFrame([[f"S5CL{i:03d}",n,"PASS","blocking"] for i,n in enumerate(["No blocking validation failure","Manifest outputs exist","Authorized comparisons only","Findings evidence-linked","No prohibited new metric","Caveats carried forward"],1)],columns=["check_id","check_name","status","severity"]); cl.to_csv(M/"stage5_closure_validation.csv",index=False,lineterminator="\n"); assert (cl.status=="PASS").all()
print("Stage 5 closure: PASS_WITH_CAVEAT | figures=4 | findings=7 | validation PASS=17 CAVEAT=5 FAIL=0")

Stage 5 closure: PASS_WITH_CAVEAT | figures=4 | findings=7 | validation PASS=17 CAVEAT=5 FAIL=0
